# NeurIPS Table 1: unified ID/OOD generative evaluation

Evaluate every generative model on the same ID + OOD trajectory split and produce a single comparable table:

- **VAE** (PINC)
- **VQ-VAE** (PINC, random codes from empirical codebook prior)
- **VQ-VAE + AR** (PINC, transformer over VQ codes)
- **Diff** (flow-matching) — 5D + linear probes (probe ae / probe gen)
- **GyroSwin** *(optional)* — set `GYROSWIN_CHECKPOINT = None` to skip

In [21]:
%load_ext autoreload
%autoreload 2

import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path

PROJECT_ROOT = "/home/u6eb/gutenbru.u6eb/plasmamodelling"
for _p in (PROJECT_ROOT, os.path.join(PROJECT_ROOT, "notebooks")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import torch
from notebooks.neurips_generate_table1 import (
    TRAJECTORIES_ID, TRAJECTORIES_OOD,
    evaluate_vae, evaluate_vqvae, evaluate_ar,
    evaluate_diff, evaluate_gyroswin,
    build_summary_table,
)

torch.use_deterministic_algorithms(False)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CUDA_VISIBLE_DEVICES = {os.environ.get('CUDA_VISIBLE_DEVICES')!r}")
print(f"torch.cuda.device_count() = {torch.cuda.device_count()}")
print(f"DEVICE = {DEVICE}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
CUDA_VISIBLE_DEVICES = '3'
torch.cuda.device_count() = 1
DEVICE = cuda


In [22]:
# ---- paths -----------------------------------------------------------
VAE_CKPT_DIR        = "/projects/u6eb/outputs/20260415_155440_886"
VQVAE_CKPT_DIR      = "/scratch/u6eb/fpaische.u6eb/outputs/20260409_111450_134"
AR_CKPT_DIR         = "/projects/u6eb/outputs/20260416_130636_591"
DIFF_CKPT_DIR       = "/projects/u6eb/outputs/20260412_180101_948/"
AE_CHECKPOINT       = "/projects/u6eb/outputs/20260405_022851_327/best.pth"
VQ_INDEX_PKL        = "/projects/u6eb/gyrokinetics/preprocessed_kvikio/diff_train_indices_offset80_mu_57f5caf20c1e_indices_vqvae134.pkl"
INFERENCE_CFG       = "/home/u6eb/gutenbru.u6eb/plasmamodelling/configs/pinc_inference.yaml"
DATA_PREP           = Path("/projects/u6eb/gyrokinetics/preprocessed_kvikio")

# Set to None to skip the GyroSwin chapter entirely.
# GYROSWIN_CHECKPOINT = "/projects/u6eb/checkpoints/gyroswin_xxl_fluxavg_cond_nodrop_l1"
GYROSWIN_CHECKPOINT = None

# ---- knobs (shared across models) ------------------------------------
N_SAMPLES         = 128   # VAE / VQ-VAE / AR / Diff prior samples per traj
BATCH_SIZE        = 64
N_DENOISING_STEPS = 15    # diff (flow-matching) inference steps
GYROSWIN_N_STEPS  = 128   # AR rollout length

# ---- cache layout ----------------------------------------------------
RESULTS_DIR = Path(PROJECT_ROOT) / "notebooks" / "results" / "table1"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

VAE_CACHE      = RESULTS_DIR / f"vae_n{N_SAMPLES}.pt"
VQVAE_CACHE    = RESULTS_DIR / f"vqvae_n{N_SAMPLES}.pt"
AR_CACHE       = RESULTS_DIR / f"ar_n{N_SAMPLES}.pt"
DIFF_CACHE     = RESULTS_DIR / f"diff_n{N_SAMPLES}_steps{N_DENOISING_STEPS}.pt"
GYROSWIN_CACHE = RESULTS_DIR / f"gyroswin_n{GYROSWIN_N_STEPS}.pt"

all_results = {}
print("Trajectories  ID :", len(TRAJECTORIES_ID))
print("Trajectories OOD :", len(TRAJECTORIES_OOD))
print("Cache dir        :", RESULTS_DIR)

Trajectories  ID : 6
Trajectories OOD : 5
Cache dir        : /home/u6eb/gutenbru.u6eb/plasmamodelling/notebooks/results/table1


## VAE

In [23]:
all_results.update(evaluate_vae(
    VAE_CKPT_DIR, INFERENCE_CFG,
    n_samples=N_SAMPLES, batch_size=BATCH_SIZE,
    output_path=str(VAE_CACHE),
    device=DEVICE,
))


  VAE
  GPU before: alloc=0.03G reserved=0.04G
  [cache hit ] /home/u6eb/gutenbru.u6eb/plasmamodelling/notebooks/results/table1/vae_n128.pt
  GPU after:  alloc=0.03G reserved=0.04G


## VQ-VAE

In [24]:
all_results.update(evaluate_vqvae(
    VQVAE_CKPT_DIR, INFERENCE_CFG,
    vq_index_pkl=VQ_INDEX_PKL,
    n_samples=N_SAMPLES, batch_size=BATCH_SIZE,
    output_path=str(VQVAE_CACHE),
    device=DEVICE,
))


  VQ-VAE
  GPU before: alloc=0.03G reserved=0.04G
  [cache hit ] /home/u6eb/gutenbru.u6eb/plasmamodelling/notebooks/results/table1/vqvae_n128.pt
  GPU after:  alloc=0.03G reserved=0.04G


## VQ-VAE + AR

In [25]:
all_results.update(evaluate_ar(
    AR_CKPT_DIR, INFERENCE_CFG,
    n_samples=N_SAMPLES, batch_size=8,
    output_path=str(AR_CACHE),
    device=DEVICE,
))


  AR (over VQ-VAE codes)
  GPU before: alloc=0.03G reserved=0.04G
  [cache hit ] /home/u6eb/gutenbru.u6eb/plasmamodelling/notebooks/results/table1/ar_n128.pt
  GPU after:  alloc=0.03G reserved=0.04G


## Diff (flow-matching)

Produces both `Diff 5D` rows and the linear-probe rows (`probe (ae)` /
`probe (gen)`). Probes are fit on training latents and applied to the
per-trajectory latents that this chapter samples — no second sampling pass.

In [26]:
all_results.update(evaluate_diff(
    DIFF_CKPT_DIR, AE_CHECKPOINT,
    data_path=DATA_PREP,
    n_samples=N_SAMPLES, gen_batch_size=BATCH_SIZE,
    n_denoising_steps=N_DENOISING_STEPS,
    with_probes=True,
    output_path=str(DIFF_CACHE),
    device=DEVICE,
))


  Diff (flow-matching)
  GPU before: alloc=0.03G reserved=0.04G
  [cache hit ] /home/u6eb/gutenbru.u6eb/plasmamodelling/notebooks/results/table1/diff_n128_steps15.pt
  GPU after:  alloc=0.03G reserved=0.04G


## GyroSwin (optional)

Skipped if `GYROSWIN_CHECKPOINT` is `None` in the config cell.

In [27]:
if GYROSWIN_CHECKPOINT:
    all_results.update(evaluate_gyroswin(
        GYROSWIN_CHECKPOINT, DIFF_CKPT_DIR, AE_CHECKPOINT,
        data_prep=DATA_PREP,
        n_steps=GYROSWIN_N_STEPS,
        output_path=str(GYROSWIN_CACHE),
        device=DEVICE,
    ))
else:
    print("GYROSWIN_CHECKPOINT is None  ->  skipping GyroSwin chapter.")

GYROSWIN_CHECKPOINT is None  ->  skipping GyroSwin chapter.


## Summary table

Single row per (model, split). Columns: eflux_RMSE, kxspec_RMSE,
kyspec_RMSE, fluxspec_RMSE. All entries computed via the same
`print_aggregate_metrics` formula.

In [28]:
summary = build_summary_table(all_results, with_diff_probes=True)
print(summary.to_string(float_format=lambda x: f"{x:.4g}"))
summary

                        eflux_RMSE  kxspec_RMSE  kyspec_RMSE  fluxspec_RMSE
model            split                                                     
VAE              ID          84.12    3.814e+05    9.291e+05           11.8
                 OOD         87.22    1.955e+06    4.732e+06          13.33
VQ-VAE           ID          70.78    4.074e+04    8.972e+04          11.41
                 OOD         72.41    1.216e+05    2.953e+05          12.67
AR               ID          48.71    3.768e+04    7.945e+04          8.318
                 OOD         46.59    1.462e+05    3.584e+05          8.621
Diff 5D          ID          9.564    1.907e+05    4.358e+05           2.99
                 OOD         15.33    1.647e+06    3.743e+06          6.321
Diff probe (ae)  ID          19.97         6813         2100          10.79
                 OOD         25.24         2472        712.7          12.41
Diff probe (gen) ID          18.18         7023         2113          10.89
            

eflux_RMSE   kxspec_RMSE   kyspec_RMSE  fluxspec_RMSE
model            split                                                       
VAE              ID      84.119011  3.813732e+05  9.290891e+05      11.800836
                 OOD     87.218994  1.954867e+06  4.731700e+06      13.332149
VQ-VAE           ID      70.778572  4.073936e+04  8.972230e+04      11.412821
                 OOD     72.414116  1.215885e+05  2.953364e+05      12.670815
AR               ID      48.713905  3.768222e+04  7.944526e+04       8.318424
                 OOD     46.593445  1.462410e+05  3.583645e+05       8.621374
Diff 5D          ID       9.563907  1.907395e+05  4.357643e+05       2.990152
                 OOD     15.331470  1.646880e+06  3.743485e+06       6.320708
Diff probe (ae)  ID      19.965065  6.813222e+03  2.099822e+03      10.792857
                 OOD     25.241928  2.472178e+03  7.126688e+02      12.413784
Diff probe (gen) ID      18.177559  7.022542e+03  2.113115e+03      10.889868
                 OOD     17.273827  2.995881e+03  6.036396e+02      12.414680